---
title: "File Uploads and Content-Addressed Artifacts"
description: "Carry browser file bytes through validated HTTP routes into retry-safe storage, then implement download, deletion, and restore."
categories: [software-engineering, full-stack, files, http, storage, reliability]
---

Files force the stack to handle data that does not belong in JSON event rows. This chapter adds an Attach file control, raw-body upload and download routes, and a content-addressed local store for patches, screenshots, logs, and exports. The browser reports the resulting digest; the service validates size and type; the object store makes a retried upload resolve to the same identity.


## Trace bytes separately from metadata

The browser reads the selected `File` into an `ArrayBuffer` and posts those bytes to `/api/artifacts` with its content type. FastAPI enforces a non-empty body and a 5 MiB local-course limit, then calls `AutocodeApplication.put_artifact`. The application delegates byte identity and atomic writes to `LocalArtifactStore`.

The response contains the SHA-256 digest, size, and content type. Session events should store that small reference rather than embedding bytes in SQLite or WebSocket frames.


In [1]:
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from autocode.runner import DemoAgentRunner
from autocode_service.api import create_app

with TemporaryDirectory() as directory:
    app = create_app(database_path=f"{directory}/sessions.db", runner=DemoAgentRunner())
    with TestClient(app) as client:
        first = client.post(
            "/api/artifacts",
            content=b"patch contents",
            headers={"Content-Type": "text/plain"},
        )
        retried = client.post(
            "/api/artifacts",
            content=b"patch contents",
            headers={"Content-Type": "text/plain"},
        )
        listing = client.get("/api/artifacts").json()
        downloaded = client.get(f"/api/artifacts/{first.json()['digest']}")

assert first.status_code == 201
assert first.json()["digest"] == retried.json()["digest"]
assert len(listing) == 1
assert downloaded.content == b"patch contents"
print("artifact digest:", first.json()["digest"])


artifact digest: ab3a636405ece190b645ffdbd50a2c46e2d01eff98192b5715242378abb97f05


The retry creates one visible object because the digest is both identity and idempotency key. The HTTP test covers browser-shaped bytes, request validation, application delegation, filesystem storage, listing, and download. In a larger deployment, metadata belongs in the database while bytes move to object storage through the same reference contract.


## Deletion is a lifecycle, not one unlink

Immediate physical deletion is unsafe when a session, another device, or an in-flight export still references the digest. `DELETE /api/artifacts/{digest}` creates a tombstone and makes ordinary GET return 404. The bytes remain recoverable during a grace period, and the restore route removes the tombstone.


In [2]:
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from autocode.runner import DemoAgentRunner
from autocode_service.api import create_app

with TemporaryDirectory() as directory:
    app = create_app(database_path=f"{directory}/sessions.db", runner=DemoAgentRunner())
    with TestClient(app) as client:
        digest = client.post("/api/artifacts", content=b"evidence").json()["digest"]
        deleted = client.delete(f"/api/artifacts/{digest}")
        hidden = client.get(f"/api/artifacts/{digest}")
        restored = client.post(f"/api/artifacts/{digest}/restore")
        visible = client.get(f"/api/artifacts/{digest}")

assert deleted.status_code == 204
assert hidden.status_code == 404
assert restored.json()["digest"] == digest
assert visible.content == b"evidence"
print("delete -> restore statuses:", deleted.status_code, hidden.status_code, restored.status_code)


delete -> restore statuses: 204 404 200


The browser receives an honest not-found state while recovery remains possible. A retention worker may physically remove old tombstoned bytes only after checking every live reference, backup policy, and grace-period clock. Chapter 09 moves that sweep out of the user request path.


## Choose proxy or direct transfer deliberately

The local service proxies bytes because it keeps the entire course runnable with one process. Object storage changes the efficient path: the API authenticates the request and mints a short-lived upload or download URL, while the browser transfers bytes directly to the object service. The session still stores the digest and metadata returned after verification.

Validate declared type, observed type when practical, length, digest, authorization, and filename display separately. Never trust a browser filename as a filesystem path, and never render uploaded HTML under the application's trusted origin.


In [3]:
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from autocode.runner import DemoAgentRunner
from autocode_service.api import create_app

with TemporaryDirectory() as directory:
    app = create_app(database_path=f"{directory}/sessions.db", runner=DemoAgentRunner())
    with TestClient(app) as client:
        empty = client.post("/api/artifacts", content=b"")
        stored = client.post(
            "/api/artifacts",
            content=b"plain text",
            headers={"Content-Type": "text/plain"},
        )

assert empty.status_code == 422
assert stored.status_code == 201
assert stored.json()["content_type"] == "text/plain"
print("validation statuses:", empty.status_code, stored.status_code)


validation statuses: 422 201


The empty-body rejection happens before storage, while content type remains metadata rather than permission to execute bytes. The capstone should add browser upload evidence, retry count, stored object count, and failed download behavior to the release scorecard.


## Exercises

Design a retention sweep for tombstones older than a grace period. List the evidence it must check before deleting bytes, and write the idempotency rule that makes running the sweep twice safe.

### [P05.1] Design a retention sweep

State the checks required before physically deleting a tombstoned artifact and the property that makes the sweep safe to retry.

In [4]:
#| echo: false
#| eval: false
#| output: false
# Purpx gung gur gbzofgbar vf byqre guna gur tenpr crevbq, ab yvir frffvba be rkcbeg ersreraprf gur qvtrfg, naq gur qryrgvba vf erpbeqrq va na nhqvg ybt. Qryrgr ol qvtrfg naq znxr gur bcrengvba vqrzcbgrag: n zvffvat bowrpg vf nyernql gur qrfverq grezvany fgngr. Onpxhcf naq bgure qrivpr ersreraprf zhfg or vapyhqrq va gur ersrerapr purpx.